In [ ]:
!pip install faiss-cpu langchain-huggingface langchain_community sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 65.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.5/443.5 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 664.8/664.8 MB 198.3 MB/s eta 0:00:0100:01

In [21]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

In [1]:
# If needed:
# !pip install -q charset-normalizer langchain-community

from charset_normalizer import from_path
import pandas as pd

csv_path = "/kaggle/input/q-ands-a/codebasics_faqs.csv"

# 1) Detect encoding
enc = (from_path(csv_path).best() or from_path(csv_path).first()).encoding or "latin1"

# 2) Read CSV (handle both new/old pandas)
read_kwargs = {"encoding": enc}
try:
    df = pd.read_csv(csv_path, on_bad_lines="skip", **read_kwargs)
except TypeError:
    # for older pandas
    df = pd.read_csv(csv_path, error_bad_lines=False, **read_kwargs)

# 3) Turn into Documents
try:
    from langchain_community.document_loaders import DataFrameLoader
except ImportError:
    # Fallback if langchain_community isn't available
    from langchain.document_loaders import DataFrameLoader

loader = DataFrameLoader(df, page_content_column="prompt")  # <- use your text column
docs = loader.load()

docs[:2]  # quick peek


ModuleNotFoundError: Module langchain_community.document_loaders not found. Please install langchain-community to access this module. You can install it using `pip install -U langchain-community`

In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectordb = FAISS.from_documents(documents=docs, embedding=embeddings)


E0000 00:00:1755759625.953311      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:230


In [27]:
retiriver = vectordb.as_retriever()
rdocs = retiriver.get_relevant_documents("How long the course is valid")
rdocs

/tmp/ipykernel_36/1787041382.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rdocs = retiriver.get_relevant_documents("How long the course is valid")


[Document(id='fd8e3aa1-e586-4bba-82b4-4209804fa89a', metadata={'response': 'Yes'}, page_content='Once purchased, is this course available for lifetime access?'),
 Document(id='1f3cf46f-77c6-4bb2-8cad-5ea0530e85b6', metadata={'response': 'We created a much lighter version of this course on YouTube available for free (click this link) and many people gave us feedback that they were able to fetch jobs (see testimonials). Now this paid course is at least 5x better than the YouTube course which gives us ample confidence that you will be able to get a job. However, we want to be honest and do not want to make any impractical promises! Our guarantee is to prepare you for the job market by teaching the most relevant skills, knowledge & timeless principles good enough to fetch the job.'}, page_content='Will this course guarantee me a job?'),
 Document(id='5f2b467d-af27-45a7-b277-1910978bfc5c', metadata={'response': 'The only prerequisite is that you need to have a functional laptop with at leas

In [1]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

NameError: name 'AutoTokenizer' is not defined

In [33]:
from langchain_community.chains import RetrievalQA
chain = RetrievalQA.from_chain_type(llm = model, chain_type = "stuff", retriever = retiriver, input_key = "query", return_source_documents = True)

ImportError: cannot import name 'RetrievalQA' from 'langchain_community.chains' (/usr/local/lib/python3.11/dist-packages/langchain_community/chains/__init__.py)

In [ ]:
chain("Do you offer EMI payments")